In [3]:

import os


os.getcwd()

'd:\\JoshFile\\Github\\face-recognition\\notebooks'

In [6]:
import sys
from pathlib import Path

root = Path.cwd().parent

if str(root) not in sys.path:
    sys.path.append(str(root))

import numpy as np
import matplotlib.pyplot as plt
# from face_recognition_api.app.core.db import SessionLocal
from face_recognition.aura_face import create_embedding
# from face_recognition_api.app.db.user import create_user
# from face_recognition_api.app.db.user import get_user_by_embedding
from PIL import Image 


d:\JoshFile\Github\face-recognition\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files: 100%|██████████| 8/8 [00:08<00:00,  1.05s/it]
d:\JoshFile\Github\face-recognition\.venv\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .\models\auraface\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .\models\auraface\2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .\models\auraface\genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .\models\auraface\glintr100.onnx recognition ['None', 3, 112, 112] 127.5 127.5
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: .\models\auraface\scrfd_10g_bnkps.onnx detection [1, 3, '?', '?'] 127.5 128.0
set det-size: (640, 640)


'd:\\JoshFile\\Github\\face-recognition\\notebooks'

In [10]:
twin_a_img_1 = Image.open("./twins/a_1.png")
twin_a_img_2 = Image.open("./twins/a_2.png")
twin_b_img_1 = Image.open("./twins/b_1.png")
twin_b_img_2 = Image.open("./twins/b_2.png")

In [12]:
# Compute embeddings for each image
def _to_vec(emb):
    vec = np.asarray(emb, dtype=np.float32).reshape(-1)
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm  # normalize for stable cosine/euclidean comparisons

def get_embedding_safe(img):
    # Try direct PIL image first; fallback to RGB numpy array
    try:
        return _to_vec(create_embedding(img))
    except Exception:
        return _to_vec(create_embedding(np.array(img.convert("RGB"))))

embeddings = {
    "a1": get_embedding_safe(twin_a_img_1),
    "a2": get_embedding_safe(twin_a_img_2),
    "b1": get_embedding_safe(twin_b_img_1),
    "b2": get_embedding_safe(twin_b_img_2),
}

def cosine_similarity(v1, v2):
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

def euclidean_distance(v1, v2):
    return float(np.linalg.norm(v1 - v2))

# Pairwise report
names = list(embeddings.keys())
print("Pairwise similarity/distance:")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        n1, n2 = names[i], names[j]
        cos = cosine_similarity(embeddings[n1], embeddings[n2])
        euc = euclidean_distance(embeddings[n1], embeddings[n2])
        print(f"{n1} vs {n2} -> cosine: {cos:.4f}, euclidean: {euc:.4f}")

# Same-person vs twin-cross comparison
same_pairs = [("a1", "a2"), ("b1", "b2")]
cross_pairs = [("a1", "b1"), ("a1", "b2"), ("a2", "b1"), ("a2", "b2")]

same_cos = [cosine_similarity(embeddings[x], embeddings[y]) for x, y in same_pairs]
same_euc = [euclidean_distance(embeddings[x], embeddings[y]) for x, y in same_pairs]
cross_cos = [cosine_similarity(embeddings[x], embeddings[y]) for x, y in cross_pairs]
cross_euc = [euclidean_distance(embeddings[x], embeddings[y]) for x, y in cross_pairs]

print("\nSummary:")
print(f"Same-person cosine avg: {np.mean(same_cos):.4f}")
print(f"Twin-cross cosine avg : {np.mean(cross_cos):.4f}")
print(f"Same-person euclid avg: {np.mean(same_euc):.4f}")
print(f"Twin-cross euclid avg : {np.mean(cross_euc):.4f}")

# Practical interpretation (tunable thresholds; model-dependent)
COSINE_SAME_THRESHOLD = 0.70
EUCLIDEAN_SAME_THRESHOLD = 0.80

print("\nPer-pair same-person decision (threshold-based):")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        n1, n2 = names[i], names[j]
        cos = cosine_similarity(embeddings[n1], embeddings[n2])
        euc = euclidean_distance(embeddings[n1], embeddings[n2])
        same_by_cos = cos >= COSINE_SAME_THRESHOLD
        same_by_euc = euc <= EUCLIDEAN_SAME_THRESHOLD
        print(
            f"{n1} vs {n2}: "
            f"same_by_cos={same_by_cos}, same_by_euc={same_by_euc}"
        )

d:\JoshFile\Github\face-recognition\.venv\Lib\site-packages\insightface\utils\face_align.py:23: FutureWarning: `estimate` is deprecated since version 0.26 and will be removed in version 2.2. Please use `SimilarityTransform.from_estimate` class constructor instead.
  tform.estimate(lmk, dst)


Pairwise similarity/distance:
a1 vs a2 -> cosine: 0.8334, euclidean: 0.5772
a1 vs b1 -> cosine: 0.7511, euclidean: 0.7055
a1 vs b2 -> cosine: 0.7322, euclidean: 0.7319
a2 vs b1 -> cosine: 0.7276, euclidean: 0.7381
a2 vs b2 -> cosine: 0.6784, euclidean: 0.8020
b1 vs b2 -> cosine: 0.8564, euclidean: 0.5360

Summary:
Same-person cosine avg: 0.8449
Twin-cross cosine avg : 0.7223
Same-person euclid avg: 0.5566
Twin-cross euclid avg : 0.7444

Per-pair same-person decision (threshold-based):
a1 vs a2: same_by_cos=True, same_by_euc=True
a1 vs b1: same_by_cos=True, same_by_euc=True
a1 vs b2: same_by_cos=True, same_by_euc=True
a2 vs b1: same_by_cos=True, same_by_euc=True
a2 vs b2: same_by_cos=False, same_by_euc=False
b1 vs b2: same_by_cos=True, same_by_euc=True
